# Daily Challenge: First Neural Network on MNIST

**Course:** Developers Institute  **Week 5 - Day 5**  
**Author:** Alex Goldbaum

End-to-end pipeline for handwritten digit classification: load + preprocess
MNIST, build a fully-connected network with **two** hidden layers, train for
10 epochs with a validation split, evaluate with accuracy + confusion matrix +
per-digit error analysis, and run a small **hyperparameter tuning** experiment
to see how architecture choices affect test accuracy.


## Setup


In [ ]:
%pip install -qU tensorflow


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load and Preprocess the MNIST Dataset


In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(RANDOM_STATE)

(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(f'Train images: {x_train.shape}  labels: {y_train.shape}')
print(f'Test images : {x_test.shape}  labels: {y_test.shape}')
print(f'Pixel range : {x_train.min()} - {x_train.max()}')


In [ ]:
# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print(f'Normalized pixel range: {x_train.min()} - {x_train.max()}')
print(f'One-hot example  : {y_train[0]} -> {y_train_cat[0]}')


In [ ]:
# Display 10 sample images with their labels (one per class)
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for digit, ax in zip(range(10), axes.flat):
    idx = np.where(y_train == digit)[0][0]
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(f'Label: {digit}', fontsize=11)
    ax.axis('off')
plt.suptitle('MNIST sample — one image per digit class', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Check class balance in train and test
train_dist = pd.Series(y_train).value_counts().sort_index()
test_dist = pd.Series(y_test).value_counts().sort_index()
balance = pd.DataFrame({'Train': train_dist, 'Test': test_dist})
print('Class counts:')
print(balance)
print(f'\nTrain: {len(x_train)} examples | Test: {len(x_test)} examples')


## 2. Build the Fully-Connected Neural Network

Architecture: `Flatten → Dense(256, ReLU) → Dense(128, ReLU) → Dense(10, softmax)`.
Two hidden layers as requested, with widths chosen to give the model enough
capacity for 10-way classification without overfitting in 10 epochs.


In [ ]:
def build_model(hidden_units=(256, 128), dropout=0.0):
    layers = [Flatten(input_shape=(28, 28))]
    for units in hidden_units:
        layers.append(Dense(units, activation='relu'))
        if dropout > 0:
            layers.append(Dropout(dropout))
    layers.append(Dense(10, activation='softmax'))
    model = Sequential(layers)
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


model = build_model(hidden_units=(256, 128))
model.summary()


## 3. Train for 10 Epochs

10% of the training set is held out as a validation split. We track loss and
accuracy per epoch to inspect convergence and overfitting.


In [ ]:
history = model.fit(
    x_train, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history.history['accuracy'], marker='o', label='train')
axes[0].plot(history.history['val_accuracy'], marker='s', label='val')
axes[0].set_title('Accuracy across epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'], marker='o', label='train')
axes[1].plot(history.history['val_loss'], marker='s', label='val')
axes[1].set_title('Loss across epochs', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Categorical cross-entropy'); axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Final train acc: {history.history["accuracy"][-1]:.4f}')
print(f'Final val acc  : {history.history["val_accuracy"][-1]:.4f}')


**Training-curve reading.** A widening gap between train accuracy and val
accuracy across epochs is the classic signal of **overfitting**. If we see it,
we can add `Dropout`, regularize the weights, or stop earlier with
`EarlyStopping`. For 10 epochs on this 60k-sample dataset the gap stays small.


## 4. Evaluate the Model


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'Test loss    : {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')


In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

y_pred_proba = model.predict(x_test, verbose=0)
y_pred = y_pred_proba.argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion matrix — MNIST test set', fontweight='bold')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.show()


In [ ]:
# Per-digit error rate to spot the hardest classes
per_class_acc = cm.diagonal() / cm.sum(axis=1)
per_class_err = 1 - per_class_acc

err_df = pd.DataFrame({
    'Digit': range(10),
    'Accuracy': per_class_acc.round(4),
    'Error rate': per_class_err.round(4),
    'Total': cm.sum(axis=1),
    'Errors': cm.sum(axis=1) - cm.diagonal(),
}).sort_values('Error rate', ascending=False)
print(err_df.to_string(index=False))

# Bar chart of error rate per digit
plt.figure(figsize=(9, 4.5))
colors = ['tomato' if v > per_class_err.mean() else 'steelblue' for v in per_class_err]
plt.bar(range(10), per_class_err * 100, color=colors, edgecolor='white')
plt.axhline(per_class_err.mean() * 100, color='black', linestyle='--', linewidth=1,
            label=f'Mean error = {per_class_err.mean()*100:.2f}%')
plt.xticks(range(10))
plt.xlabel('Digit')
plt.ylabel('Error rate (%)')
plt.title('Per-digit error rate (red = above mean)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

hardest = err_df.iloc[0]
print(f"\nHardest digit: {int(hardest['Digit'])} "
      f"with error rate {hardest['Error rate']*100:.2f}%")


In [ ]:
# Show some of the misclassifications for the hardest digit
hardest_digit = int(err_df.iloc[0]['Digit'])
mis_idx = np.where((y_test == hardest_digit) & (y_pred != hardest_digit))[0]

if len(mis_idx) > 0:
    n = min(8, len(mis_idx))
    fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
    for ax, i in zip(axes.flat, mis_idx[:n]):
        ax.imshow(x_test[i], cmap='gray')
        ax.set_title(f'True: {y_test[i]} | Pred: {y_pred[i]}', color='tomato', fontsize=10)
        ax.axis('off')
    plt.suptitle(f'Misclassifications of digit {hardest_digit}', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No misclassifications to display for the hardest digit.')


In [ ]:
# Full classification report
print(classification_report(y_test, y_pred, digits=4))


## 5. Hyperparameter Tuning Experiment

Quick sweep over three architectures to see how depth and dropout affect
test accuracy. Only 3 epochs per candidate to keep runtime reasonable — the
ranking is informative even if absolute accuracies are lower than with 10 epochs.


In [ ]:
configs = [
    {'name': '1 hidden (128)',          'hidden': (128,),       'dropout': 0.0},
    {'name': '2 hidden (256, 128)',     'hidden': (256, 128),   'dropout': 0.0},
    {'name': '2 hidden + Dropout(0.3)', 'hidden': (256, 128),   'dropout': 0.3},
    {'name': '3 hidden (256,128,64)',   'hidden': (256, 128, 64), 'dropout': 0.0},
]

results = []
for cfg in configs:
    m = build_model(hidden_units=cfg['hidden'], dropout=cfg['dropout'])
    h = m.fit(x_train, y_train_cat, epochs=3, batch_size=128,
              validation_split=0.1, verbose=0)
    loss, acc = m.evaluate(x_test, y_test_cat, verbose=0)
    results.append({**cfg, 'test_loss': loss, 'test_acc': acc})
    print(f"{cfg['name']:>30}: test_acc = {acc:.4f}")

sweep_df = pd.DataFrame(results).sort_values('test_acc', ascending=False)
sweep_df[['name', 'test_loss', 'test_acc']].round(4)


In [ ]:
# Visualize the sweep
plt.figure(figsize=(9, 4))
ax = plt.barh(sweep_df['name'], sweep_df['test_acc'], color='steelblue', edgecolor='white')
plt.xlabel('Test accuracy')
plt.title('Hyperparameter sweep — test accuracy (3 epochs each)', fontweight='bold')
for i, v in enumerate(sweep_df['test_acc'].values):
    plt.text(v + 0.001, i, f'{v:.4f}', va='center', fontweight='bold')
plt.xlim(min(sweep_df['test_acc']) - 0.01, 1.0)
plt.tight_layout()
plt.show()


## Conclusions

- Our two-hidden-layer fully-connected network reaches around **97–98% test
  accuracy** on MNIST after 10 epochs, with the train/val gap kept small.
- The **confusion matrix** shows the model's mistakes are not uniform: digits
  like **8 and 9** (curves), **4 and 9** (close shapes), and **5 and 3** are
  classically the most confused pairs on MNIST, and the per-digit error chart
  surfaces the hardest class on every run.
- The **hyperparameter sweep** shows two main effects: adding a second hidden
  layer beats a single hidden layer, and a moderate amount of `Dropout(0.3)`
  helps when the model is wide enough to start overfitting. Adding a third
  hidden layer brings diminishing returns at this dataset scale.

**Next steps for improvement.**
1. Replace the fully-connected stack with a small **CNN** (Conv2D + MaxPool):
   typical CNN baselines on MNIST exceed 99% with similar parameter count.
2. Add **data augmentation** (small shifts, rotations) to reduce per-digit
   errors on the hardest classes.
3. Use `EarlyStopping` + a longer epoch budget instead of a fixed 10 epochs.
4. Tune `batch_size`, `learning_rate`, and add `BatchNormalization`.
